In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score
import numpy as np

In [ ]:
# =============================
# 1. Load dataset
# =============================
train_df = pd.read_csv("/content/drive/MyDrive/seminar 2/dataset/train.csv")  # Kaggle Toxic Comment dataset
# train.csv có các cột: id, comment_text, toxic, severe_toxic, obscene, threat, insult, identity_hate

LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
num_labels = len(LABELS)

In [ ]:
torch.tensor(train_df[LABELS].iloc[0].values, dtype=torch.float)

tensor([0., 0., 0., 0., 0., 0.])

In [ ]:
# =============================
# 2. Dataset class
# =============================
class ToxicDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = str(self.df["comment_text"].iloc[idx])
        inputs = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        labels = torch.tensor(self.df[LABELS].iloc[idx].values, dtype=torch.float)
        return {
            "input_ids": inputs["input_ids"].squeeze(),
            "attention_mask": inputs["attention_mask"].squeeze(),
            "labels": labels
        }

In [ ]:
#=============================
# 3. Tokenizer + Model
# =============================
# 'answerdotai/ModernBERT-base''FacebookAI/roberta-base'
MODEL_NAME = "FacebookAI/roberta-base"   # hoặc "bert-base-uncased", "roberta-base", "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels, problem_type="multi_label_classification")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# =============================
# 4. DataLoader
# =============================
train_dataset = ToxicDataset(train_df, tokenizer, max_len=128)

# Split 8:1 train-validation
train_size = int( len(train_dataset)/9*8)
val_size = len(train_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(train_dataset, [train_size, val_size])

In [ ]:
# =============================
# 5. Metrics
# =============================
def multi_label_metrics(predictions, labels, threshold=0.5):
    preds = (predictions >= threshold).astype(int)
    acc = (preds == labels).mean()
    f1_micro = f1_score(labels, preds, average="micro")
    f1_macro = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_micro": f1_micro, "f1_macro": f1_macro}
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = 1 / (1 + np.exp(-logits))  # sigmoid
    return multi_label_metrics(predictions, labels)

In [ ]:
# =============================
# 6. Training
# =============================
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/seminar 2/toxic_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=5000,
    load_best_model_at_end=True,
    save_total_limit=2,
    report_to =None,
    metric_for_best_model ='eval_f1_macro'

)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-4015516837.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: bangtt to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1 Micro,F1 Macro
1,0.053300,0.038800,0.985273,0.783144,0.634869
2,0.036200,0.039055,0.984573,0.795571,0.625486
3,0.030200,0.041545,0.984886,0.796570,0.666248
4,0.023600,0.046090,0.985064,0.791909,0.682769
5,0.020300,0.050679,0.984845,0.793393,0.681218


TrainOutput(global_step=39895, training_loss=0.0315125406707255, metrics={'train_runtime': 15525.1603, 'train_samples_per_second': 41.113, 'train_steps_per_second': 2.57, 'total_flos': 4.198613895604224e+16, 'train_loss': 0.0315125406707255, 'epoch': 5.0})

In [ ]:
test_df = pd.read_csv("/content/drive/MyDrive/seminar 2/dataset/test.csv")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('/content/drive/MyDrive/seminar 2/checkpoint-7979')
tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/seminar 2/checkpoint-7979")

In [ ]:
# =============================
# 4. DataLoader cho test
# =============================
test_dataset = ToxicDataset(test_df, tokenizer, max_len=128)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
predictions = []
model.eval()
with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.sigmoid(logits).cpu().numpy()   # sigmoid cho multi-label
        predictions.append(probs)
predictions = np.vstack(predictions)

100%|██████████| 499/499 [1:57:23<00:00, 14.12s/it]


In [ ]:
predict=[[1 if i>=0.5 else 0 for i in _] for _ in predictions]

In [ ]:
gold=test_df[LABELS].values.tolist()

In [ ]:
from sklearn.metrics import accuracy_score
print(accuracy_score(gold,predict))

0.9486777791703221


In [ ]:
print(classification_report(gold,predict,target_names=LABELS,digits=4))

               precision    recall  f1-score   support

        toxic     0.9189    0.8827    0.9004      1501
 severe_toxic     0.5278    0.5101    0.5188       149
      obscene     0.8764    0.8677    0.8720       809
       threat     0.8077    0.4468    0.5753        47
       insult     0.8369    0.8544    0.8456       769
identity_hate     0.6358    0.8136    0.7138       118

    micro avg     0.8591    0.8479    0.8535      3393
    macro avg     0.7672    0.7292    0.7377      3393
 weighted avg     0.8616    0.8479    0.8535      3393
  samples avg     0.0771    0.0797    0.0766      3393



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
